In [58]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

files = [
    Path(f) for f in glob.glob("sim_stats/*/*/*/ue_stats.csv")
    if 1 <= int(Path(f).parent.name.removeprefix("SEED")) <= 10
]
dfs = []

for file in files:
    df = pd.read_csv(file)
    df["policy"] = file.parents[2].name
    df["scenario"] = file.parents[1].name
    dfs.append(df)

if not dfs:
    raise ValueError("No ue_stats.csv files matched the selected policies and seeds")
df = pd.concat(dfs, ignore_index=True)

required = {"completed_packets", "dropped_packets", "delay_budget"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"ue_stats.csv missing columns: {sorted(missing)}")

packet_counts = df[["completed_packets", "dropped_packets"]]
if not np.isfinite(packet_counts.to_numpy(dtype=float)).all():
    raise ValueError("packet counts must contain only finite values")
if (packet_counts < 0).any().any():
    raise ValueError("packet counts cannot be negative")

df["total_packets"] = (
    df["completed_packets"] + df["dropped_packets"]
).astype(int)

# A UE leaves the simulator only after its packets have outcomes.  The number
# of packets per UE is configurable, so aggregate the recorded packet counts.
if (df["total_packets"] <= 0).any():
    raise ValueError("Each completed UE must contain at least one packet outcome")

stats = (
    df.groupby(["policy", "scenario", "delay_budget"])
    .agg(
        total_ues=("total_packets", "size"),
        total_packets=("total_packets", "sum"),
        completed_packets=("completed_packets", "sum"),
        dropped_packets=("dropped_packets", "sum"),
    )
)

# Percentage, not a fraction.
stats["drop_ratio"] = (
    100 * stats["dropped_packets"] / stats["total_packets"]
)

print(stats.to_string(float_format=lambda x: f"{x:.4f}"))


                                                                  total_ues  total_packets  completed_packets  dropped_packets  drop_ratio
policy                                     scenario delay_budget                                                                          
A_w_str=150.0_w_sen=1.5_w_tol=1.5_w_pc=1.0 A        0.05                 10            500                497                3      0.6000
                                                    0.15               1018          50900              50712              188      0.3694
                                                    0.30               1661          83050              82883              167      0.2011


In [59]:
from pathlib import Path
import numpy as np
import pandas as pd

policy = "A*"


files = sorted(
    Path("sim_stats").glob(f"{policy}/*/*/bs_stats.csv"),
    key=lambda p: int(p.parent.name.removeprefix("SEED"))
)

rows = []
all_times = np.zeros(4)

for file in files:
    df = pd.read_csv(file)

    # sleep_time 顺序：Active、SM1、SM2、SM3
    times = np.stack([
        np.fromstring(str(value).strip("[]"), sep=" ")
        for value in df["sleep_time"]
    ])

    if times.shape[1] != 4 or not np.isfinite(times).all():
        raise ValueError(f"sleep_time 格式错误：{file}")
    if (times < 0).any() or times.sum() <= 0:
        raise ValueError(f"sleep_time 数值错误：{file}")
    # 汇总所有基站的驻留时间，再计算比例
    seed_times = times.sum(axis=0)
    all_times += seed_times

    rows.append([
        file.parent.name,
        *(seed_times / seed_times.sum() * 100)
    ])

rows.append(["整体", *(all_times / all_times.sum() * 100)])

result = pd.DataFrame(
    rows,
    columns=["seed", "Active (%)", "SM1 (%)", "SM2 (%)", "SM3 (%)"]
).set_index("seed")

print(result.to_string(float_format=lambda x: f"{x:.3f}"))

       Active (%)  SM1 (%)  SM2 (%)  SM3 (%)
seed                                        
SEED1      34.797    0.001   47.021   18.181
SEED2      35.958    0.000   46.576   17.465
整体         35.377    0.001   46.798   17.823
